In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Data Aggregation").getOrCreate()

26/05/03 18:45:02 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

In [2]:
listings = spark.read.csv("listings.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")


In [3]:
listings.groupby(listings.property_type).count().show(truncate=False)

[Stage 4:>                                                          (0 + 1) / 1]

+----------------------------------+-----+
|property_type                     |count|
+----------------------------------+-----+
|Private room in lighthouse        |2    |
|Private room in loft              |153  |
|Private room in earthen home      |2    |
|Entire chalet                     |4    |
|Earthen home                      |1    |
|Farm stay                         |4    |
|Entire rental unit                |41215|
|Shared room in hostel             |66   |
|Shared room                       |1    |
|Private room in condo             |3189 |
|Room in boutique hotel            |217  |
|Private room in religious building|4    |
|Room in bed and breakfast         |18   |
|Private room in casa particular   |56   |
|Private room in bungalow          |63   |
|Entire cabin                      |43   |
|Entire guesthouse                 |221  |
|Hut                               |3    |
|Private room in nature lodge      |4    |
|Entire guest suite                |177  |
+----------

In [4]:
import pyspark.sql.functions as F

listings.groupby(listings.property_type).agg(F.count('property_type').alias('count')).orderBy('count', ascending=[False]).show(truncate=False)

[Stage 7:>                                                          (0 + 1) / 1]

+----------------------------------+-----+
|property_type                     |count|
+----------------------------------+-----+
|Entire rental unit                |41215|
|Private room in rental unit       |14464|
|Private room in home              |11704|
|Entire home                       |9120 |
|Entire condo                      |8250 |
|Private room in condo             |3189 |
|Entire serviced apartment         |1874 |
|Private room in townhouse         |1195 |
|Room in hotel                     |1113 |
|Entire townhouse                  |1058 |
|Private room in bed and breakfast |491  |
|Private room in guesthouse        |377  |
|Entire loft                       |341  |
|Entire guesthouse                 |221  |
|Room in boutique hotel            |217  |
|Entire guest suite                |177  |
|Private room in guest suite       |174  |
|Private room in loft              |153  |
|Private room in serviced apartment|132  |
|Private room                      |103  |
+----------

In [5]:
listings.groupby(listings.property_type).agg(F.count('property_type').alias('count'), F.avg('review_scores_location')).orderBy('count', ascending=[False]).show(truncate=False)

[Stage 10:>                                                         (0 + 1) / 1]

+----------------------------------+-----+---------------------------+
|property_type                     |count|avg(review_scores_location)|
+----------------------------------+-----+---------------------------+
|Entire rental unit                |41215|4.727437046043664          |
|Private room in rental unit       |14464|4.726647667299953          |
|Private room in home              |11704|4.694735943407702          |
|Entire home                       |9120 |4.727462068965489          |
|Entire condo                      |8250 |4.770284788770394          |
|Private room in condo             |3189 |4.778348954578206          |
|Entire serviced apartment         |1874 |4.722610024449879          |
|Private room in townhouse         |1195 |4.766042471042479          |
|Room in hotel                     |1113 |4.608689075630254          |
|Entire townhouse                  |1058 |4.81762931034483           |
|Private room in bed and breakfast |491  |4.724728915662653          |
|Priva

In [6]:
reviews = spark.read.csv("reviews.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")


In [7]:
for field in reviews.schema:
    print(field)

StructField('listing_id', LongType(), True)
StructField('id', LongType(), True)
StructField('date', DateType(), True)
StructField('reviewer_id', IntegerType(), True)
StructField('reviewer_name', StringType(), True)
StructField('comments', StringType(), True)


In [11]:
listings_reviews = listings.join(reviews, listings.id == reviews.listing_id, how='inner')

In [9]:
listings_reviews.groupBy('id').agg(F.count('id').alias('num_reviews')).show()

{"ts": "2026-05-03 18:55:21.310", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `id` is ambiguous, could be: [`id`, `id`]. SQLSTATE: 42704", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "AMBIGUOUS_REFERENCE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o126.agg.\n: org.apache.spark.sql.AnalysisException: [AMBIGUOUS_REFERENCE] Reference `id` is ambiguous, could be: [`id`, `id`]. SQLSTATE: 42704\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:2232)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolve(package.scala:356)\n\tat org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.resolveChildren(LogicalPlan.scala:164)\n\tat org.apache.spark.sql.catalyst.analysis.ColumnResolutionHelper.$anonfun$resolveExpressionByP

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `id` is ambiguous, could be: [`id`, `id`]. SQLSTATE: 42704

In [10]:
reviews_per_listing = listings_reviews.groupBy(listings.id, listings.name).agg(F.count(reviews.id).alias('num_reviews')).orderBy('num_reviews', ascending=False).show(truncate=False)

NameError: name 'listing_reviews' is not defined